In [41]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''
Created on 2024-08-14
Last modified on 2024-08-14
@author: Juan Enrique López
@description: Jupyter Notebook cuyo cometido es generar archivos md con la información relativa a las relaciones entre las propiedades Mitre con la finalidad de filtrar vía Obsidian.
Por el momento solo se generan las relaciones desde la vista 1:N donde 1 es la propiedad Mitre y N las técnicas asociadas

'''

'\nCreated on 2024-08-14\nLast modified on 2024-08-14\n@author: Juan Enrique López\n@description: Jupyter Notebook cuyo cometido es generar archivos md con la información relativa a las relaciones entre las propiedades Mitre con la finalidad de filtrar vía Obsidian.\nPor el momento solo se generan las relaciones desde la vista 1:N donde 1 es la propiedad Mitre y N las técnicas asociadas\n\n'

**IMPORTANTE - REQUERIMIENTOS PREVIOS**

- Requiere haber ejecutado previamente dentro de este mismo módulo **mitre_relationships** el notebook **[stix2]_mitre_relationships.ipynb** y disponer de la carpeta "outputs/stix2". 


In [42]:
import os
import pandas as pd
import shutil
# import requests
from stix2 import Filter, MemoryStore

#### **Parámetros**

In [43]:
mitre_domain = 'enterprise' # enterprise, mobile, ics
input_path = os.path.join(os.getcwd(),'outputs', 'stix2', mitre_domain, 'relations')

In [44]:
output_folder = os.path.join(os.getcwd(),'outputs')
output_folder

'c:\\Users\\jelopez\\Documents\\CyberProof\\python\\develop\\mitre_relationships\\outputs'

#### **Funciones**

In [45]:
def format_cols_for_md(column):
    '''
    Función para aplicar los reemplazos que permitan en el .md relacionar elementos.
    '''
    column = column.apply(lambda x: f'[[{x}]]')
    return column

In [46]:
def replace_chars_add_bracket(column):
    '''
    Función encargada de reemplazar caracteres en una columna dada con el objetivo añadir una lista de elementos enlazados para Obsidian.
    '''
    if column.dtype == "object":  # Verifica que la columna sea de tipo string
        column = column.str.replace("['", '[[', regex=False)
        column = column.str.replace("']", ']]', regex=False)
        column = column.str.replace("', '", ']] [[', regex=False)
    return column

In [47]:
def replace_chars(column):
    '''
    Función encargada de reemplazar caracteres en una columna dada con el objetivo de suprimir caracteres que enlazan dentro de Obsidian.
    '''
    if column.dtype == "object":  # Verifica que la columna sea de tipo string
        column = column.str.replace("['", '', regex=False)
        column = column.str.replace("']", '', regex=False)
        column = column.str.replace("'", '', regex=False)
    return column

#### **Ejecución**

##### **1. Tácticas - Técnicas**

In [48]:
input_file = f'[MITRE]_{mitre_domain}_techniques_tactic_N1.csv'
tactics_relations_df = pd.read_csv(os.path.join(input_path, 'techniques_tactics', input_file), sep=';', quotechar='"')
tactics_relations_df.head(3)

,tactic_ID,tactic,technique_ID,technique
0,TA0001,['Initial Access'],"['T1566.004', 'T1078.002', 'T1190', 'T1566', '...","['Trusted Relationship', 'Drive-by Compromise'..."
1,TA0002,['Execution'],"['T1059.009', 'T1053.002', 'T1047', 'T1204.002...","['Windows Management Instrumentation', 'Deploy..."
2,TA0003,['Persistence'],"['T1546.006', 'T1098.001', 'T1078.002', 'T1574...","['Registry Run Keys / Startup Folder', 'Outloo..."


In [49]:
add_bracket = ['tactic_ID','platform_ID','platform','group_ID','data_source_ID']
replace_bracket = ['technique_ID']
for col in tactics_relations_df.columns:
    if col in add_bracket:
        print(col)
        tactics_relations_df[col] = format_cols_for_md(tactics_relations_df[col])
    elif col in replace_bracket:
        tactics_relations_df[col] = replace_chars_add_bracket(tactics_relations_df[col])
    else:
        tactics_relations_df[col] =  replace_chars(tactics_relations_df[col])
tactics_relations_df.columns = tactics_relations_df.columns.str.replace('_', ' ', regex=False)
tactics_relations_df.head(3)

tactic_ID


,tactic ID,tactic,technique ID,technique
0,[[TA0001]],Initial Access,[[T1566.004]] [[T1078.002]] [[T1190]] [[T1566]...,"Trusted Relationship, Drive-by Compromise, Spe..."
1,[[TA0002]],Execution,[[T1059.009]] [[T1053.002]] [[T1047]] [[T1204....,"Windows Management Instrumentation, Deploy Con..."
2,[[TA0003]],Persistence,[[T1546.006]] [[T1098.001]] [[T1078.002]] [[T1...,"Registry Run Keys / Startup Folder, Outlook Fo..."


In [50]:
for index, row in tactics_relations_df.iterrows():
    # Obtener el tactic_ID sin los corchetes
    tactic_id = row['tactic ID'].strip('[]')
    
    # Crear la ruta del archivo
    folder_path = os.path.join(output_folder, 'mitre properties', 'relations_md', mitre_domain, 'tactics_techniques', tactic_id)
    os.makedirs(folder_path, exist_ok=True)
    
    # Crear el nombre del archivo .md
    filename = f'relations_{tactic_id}.md'
    file_path = os.path.join(folder_path, filename)
    
    # Escribir el contenido en el archivo .md
    with open(file_path, 'w') as file:
        # Propiedades
        file.write("---\n")
        file.write("Tactic: true\n")
        file.write("Tactic relations: true\n")
        file.write("---\n\n")
        
        # Escribir el título con tactic_ID
        file.write(f"### {tactic_id}\n\n")
        
        # Escribir los detalles de la táctica
        file.write(f"**tactic ID**\n{row['tactic ID']}\n\n")
        file.write(f"**related techniques**\n{row['technique ID']}\n\n")

        file.close()
    print(f"Archivo '{filename}' creado en la carpeta '{file_path}' con éxito.")

Archivo 'relations_TA0001.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\tactics_techniques\TA0001\relations_TA0001.md' con éxito.
Archivo 'relations_TA0002.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\tactics_techniques\TA0002\relations_TA0002.md' con éxito.
Archivo 'relations_TA0003.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\tactics_techniques\TA0003\relations_TA0003.md' con éxito.
Archivo 'relations_TA0004.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\tactics_techniques\TA0004\relations_TA0004.md' con éxito.
Archivo 'relations_TA0005.md' creado en la carpeta 'c:\Users\jelopez\Doc

##### **2. Software - Técnicas**

In [51]:
input_file = f'[MITRE]_{mitre_domain}_techniques_software_N1.csv'
software_relations_df = pd.read_csv(os.path.join(input_path, 'techniques_software', input_file), sep=';', quotechar='"')
software_relations_df.head(3)

,software_ID,software,technique_ID,technique
0,S0001,['Trojan.Mebromi'],['T1542.001'],['System Firmware']
1,S0002,['Mimikatz'],"['T1003.006', 'T1550.003', 'T1552.004', 'T1003...","['Pass the Hash', 'Credentials from Password S..."
2,S0003,['RIPTIDE'],"['T1071.001', 'T1573.001']","['Web Protocols', 'Symmetric Cryptography']"


In [52]:
add_bracket = ['tactic_ID','platform_ID','platform','group_ID','data_source_ID', 'software_ID']
replace_bracket = ['technique_ID']
for col in software_relations_df.columns:
    if col in add_bracket:
        print(col)
        software_relations_df[col] = format_cols_for_md(software_relations_df[col])
    elif col in replace_bracket:
        software_relations_df[col] = replace_chars_add_bracket(software_relations_df[col])
    else:
        software_relations_df[col] =  replace_chars(software_relations_df[col])
software_relations_df.columns = software_relations_df.columns.str.replace('_', ' ', regex=False)
software_relations_df.head(3)

software_ID


,software ID,software,technique ID,technique
0,[[S0001]],Trojan.Mebromi,[[T1542.001]],System Firmware
1,[[S0002]],Mimikatz,[[T1003.006]] [[T1550.003]] [[T1552.004]] [[T1...,"Pass the Hash, Credentials from Password Store..."
2,[[S0003]],RIPTIDE,[[T1071.001]] [[T1573.001]],"Web Protocols, Symmetric Cryptography"


In [53]:
for index, row in software_relations_df.iterrows():
    # Obtener el software_id sin los corchetes
    software_id = row['software ID'].strip('[]')
    
    # Crear la ruta del archivo
    folder_path = os.path.join(output_folder, 'mitre properties', 'relations_md', mitre_domain, 'software_techniques', software_id)
    os.makedirs(folder_path, exist_ok=True)
    
    # Crear el nombre del archivo .md
    filename = f'relations_{software_id}.md'
    file_path = os.path.join(folder_path, filename)
    
    # Escribir el contenido en el archivo .md
    with open(file_path, 'w') as file:
        # Propiedades
        file.write("---\n")
        file.write("Software: true\n")
        file.write("Software relations: true\n")
        file.write("---\n\n")
        
        # Escribir el título con software_id
        file.write(f"### {software_id}\n\n")
        
        # Escribir los detalles de la táctica
        file.write(f"**software ID**\n{row['software ID']}\n\n")
        file.write(f"**related techniques**\n{row['technique ID']}\n\n")

        file.close()
    print(f"Archivo '{filename}' creado en la carpeta '{file_path}' con éxito.")

Archivo 'relations_S0001.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\software_techniques\S0001\relations_S0001.md' con éxito.
Archivo 'relations_S0002.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\software_techniques\S0002\relations_S0002.md' con éxito.
Archivo 'relations_S0003.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\software_techniques\S0003\relations_S0003.md' con éxito.
Archivo 'relations_S0004.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\software_techniques\S0004\relations_S0004.md' con éxito.
Archivo 'relations_S0005.md' creado en la carpeta 'c:\Users\jelopez\Documents\Cy

Archivo 'relations_S0008.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\software_techniques\S0008\relations_S0008.md' con éxito.
Archivo 'relations_S0009.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\software_techniques\S0009\relations_S0009.md' con éxito.
Archivo 'relations_S0010.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\software_techniques\S0010\relations_S0010.md' con éxito.
Archivo 'relations_S0011.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\software_techniques\S0011\relations_S0011.md' con éxito.
Archivo 'relations_S0012.md' creado en la carpeta 'c:\Users\jelopez\Documents\Cy

##### **4. Data Sources**

In [54]:
input_file = f'[MITRE]_{mitre_domain}_techniques_datasource_N1.csv'
datasources_relations_df = pd.read_csv(os.path.join(input_path, 'techniques_datasources', input_file), sep=';', quotechar='"')
datasources_relations_df.head(3)

,data_source_ID,data_source,technique_ID,technique
0,DS0001,['Firmware'],"['T1542.001', 'T1495', 'T1564.005', 'T1542', '...","['TFTP Boot', 'Rootkit', 'Firmware Corruption'..."
1,DS0002,['User Account'],"['T1098.001', 'T1078.002', 'T1556.005', 'T1136...","['Unsecured Credentials', 'Use Alternate Authe..."
2,DS0003,['Scheduled Job'],"['T1053.002', 'T1053.007', 'T1036.004', 'T1053...","['Masquerade Task or Service', 'At', 'Schedule..."


In [55]:
add_bracket = ['tactic_ID','platform_ID','platform','group_ID','data_source_ID', 'software_ID']
replace_bracket = ['technique_ID']
for col in datasources_relations_df.columns:
    if col in add_bracket:
        print(col)
        datasources_relations_df[col] = format_cols_for_md(datasources_relations_df[col])
    elif col in replace_bracket:
        datasources_relations_df[col] = replace_chars_add_bracket(datasources_relations_df[col])
    else:
        datasources_relations_df[col] =  replace_chars(datasources_relations_df[col])
datasources_relations_df.columns = datasources_relations_df.columns.str.replace('_', ' ', regex=False)
datasources_relations_df.head(3)

data_source_ID


,data source ID,data source,technique ID,technique
0,[[DS0001]],Firmware,[[T1542.001]] [[T1495]] [[T1564.005]] [[T1542]...,"TFTP Boot, Rootkit, Firmware Corruption, Hidde..."
1,[[DS0002]],User Account,[[T1098.001]] [[T1078.002]] [[T1556.005]] [[T1...,"Unsecured Credentials, Use Alternate Authentic..."
2,[[DS0003]],Scheduled Job,[[T1053.002]] [[T1053.007]] [[T1036.004]] [[T1...,"Masquerade Task or Service, At, Scheduled Task..."


In [56]:
for index, row in datasources_relations_df.iterrows():
    # Obtener el datasource_id sin los corchetes
    datasource_id = row['data source ID'].strip('[]')
    
    # Crear la ruta del archivo
    folder_path = os.path.join(output_folder, 'mitre properties', 'relations_md', mitre_domain, 'datasources_techniques', datasource_id)
    os.makedirs(folder_path, exist_ok=True)
    
    # Crear el nombre del archivo .md
    filename = f'relations_{datasource_id}.md'
    file_path = os.path.join(folder_path, filename)
    
    # Escribir el contenido en el archivo .md
    with open(file_path, 'w') as file:
        # Propiedades
        file.write("---\n")
        file.write("Data source: true\n")
        file.write("Data source relations: true\n")
        file.write("---\n\n")
        
        # Escribir el título con datasource_id
        file.write(f"### {datasource_id}\n\n")
        
        # Escribir los detalles de la táctica
        file.write(f"**data source ID**\n{row['data source ID']}\n\n")
        file.write(f"**related techniques**\n{row['technique ID']}\n\n")

        file.close()
    print(f"Archivo '{filename}' creado en la carpeta '{file_path}' con éxito.")

Archivo 'relations_DS0001.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\datasources_techniques\DS0001\relations_DS0001.md' con éxito.


Archivo 'relations_DS0002.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\datasources_techniques\DS0002\relations_DS0002.md' con éxito.
Archivo 'relations_DS0003.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\datasources_techniques\DS0003\relations_DS0003.md' con éxito.
Archivo 'relations_DS0004.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\datasources_techniques\DS0004\relations_DS0004.md' con éxito.
Archivo 'relations_DS0005.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\datasources_techniques\DS0005\relations_DS0005.md' con éxito.
Archivo 'relations_DS0006.md' creado en la carpeta 'c:\U

##### **5. Grupos Técnicas**

In [57]:
input_file = f'[MITRE]_{mitre_domain}_techniques_group_N1.csv'
groups_relations_df = pd.read_csv(os.path.join(input_path, 'techniques_groups', input_file), sep=';', quotechar='"')
groups_relations_df.head(3)

,group_ID,group,technique_ID,technique
0,G0001,['Axiom'],"['T1021.001', 'T1583.003', 'T1546.008', 'T1189...","['Valid Accounts', 'OS Credential Dumping', 'E..."
1,G0002,['Moafee'],['T1027.001'],['Binary Padding']
2,G0003,['Cleaver'],"['T1588.002', 'T1587.001', 'T1585.001', 'T1557...","['Malware', 'Social Media Accounts', 'LSASS Me..."


In [58]:
add_bracket = ['tactic_ID','platform_ID','platform','group_ID','data_source_ID', 'software_ID']
replace_bracket = ['technique_ID']
for col in groups_relations_df.columns:
    if col in add_bracket:
        print(col)
        groups_relations_df[col] = format_cols_for_md(groups_relations_df[col])
    elif col in replace_bracket:
        groups_relations_df[col] = replace_chars_add_bracket(groups_relations_df[col])
    else:
        groups_relations_df[col] =  replace_chars(groups_relations_df[col])
groups_relations_df.columns = groups_relations_df.columns.str.replace('_', ' ', regex=False)
groups_relations_df.head(3)

group_ID


,group ID,group,technique ID,technique
0,[[G0001]],Axiom,[[T1021.001]] [[T1583.003]] [[T1546.008]] [[T1...,"Valid Accounts, OS Credential Dumping, Exploit..."
1,[[G0002]],Moafee,[[T1027.001]],Binary Padding
2,[[G0003]],Cleaver,[[T1588.002]] [[T1587.001]] [[T1585.001]] [[T1...,"Malware, Social Media Accounts, LSASS Memory, ..."


In [59]:
for index, row in groups_relations_df.iterrows():
    # Obtener el group_id sin los corchetes
    group_id = row['group ID'].strip('[]')
    
    # Crear la ruta del archivo
    folder_path = os.path.join(output_folder, 'mitre properties', 'relations_md', mitre_domain, 'groups_techniques', group_id)
    os.makedirs(folder_path, exist_ok=True)
    
    # Crear el nombre del archivo .md
    filename = f'relations_{group_id}.md'
    file_path = os.path.join(folder_path, filename)
    
    # Escribir el contenido en el archivo .md
    with open(file_path, 'w') as file:
        # Propiedades
        file.write("---\n")
        file.write("Group: true\n")
        file.write("Group relations: true\n")
        file.write("---\n\n")
        
        # Escribir el título con group_id
        file.write(f"### {group_id}\n\n")
        
        # Escribir los detalles de la táctica
        file.write(f"**group ID**\n{row['group ID']}\n\n")
        file.write(f"**related techniques**\n{row['technique ID']}\n\n")

        file.close()
    print(f"Archivo '{filename}' creado en la carpeta '{file_path}' con éxito.")

Archivo 'relations_G0001.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\groups_techniques\G0001\relations_G0001.md' con éxito.
Archivo 'relations_G0002.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\groups_techniques\G0002\relations_G0002.md' con éxito.
Archivo 'relations_G0003.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\groups_techniques\G0003\relations_G0003.md' con éxito.
Archivo 'relations_G0004.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\groups_techniques\G0004\relations_G0004.md' con éxito.
Archivo 'relations_G0005.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof

Archivo 'relations_G0023.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\groups_techniques\G0023\relations_G0023.md' con éxito.
Archivo 'relations_G0024.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\groups_techniques\G0024\relations_G0024.md' con éxito.
Archivo 'relations_G0025.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\groups_techniques\G0025\relations_G0025.md' con éxito.
Archivo 'relations_G0026.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\groups_techniques\G0026\relations_G0026.md' con éxito.
Archivo 'relations_G0027.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof

##### **6. Plataformas - Técnicas**

In [60]:
input_file = f'[MITRE]_{mitre_domain}_techniques_platform_N1.csv'
platforms_relations_df = pd.read_csv(os.path.join(input_path, 'techniques_platforms', input_file), sep=';', quotechar='"')
platforms_relations_df.head(3)

,platform,technique_ID,technique
0,Azure AD,"['T1059.009', 'T1098.001', 'T1548', 'T1528', '...","['Unsecured Credentials', 'Conditional Access ..."
1,Containers,"['T1068', 'T1190', 'T1136.001', 'T1496', 'T152...","['Unsecured Credentials', 'Use Alternate Authe..."
2,Google Workspace,"['T1059.009', 'T1566.004', 'T1564.008', 'T1048...","['Unsecured Credentials', 'Use Alternate Authe..."


In [61]:
add_bracket = ['tactic_ID','platform_ID','platform','group_ID','data_source_ID', 'software_ID']
replace_bracket = ['technique_ID']
for col in platforms_relations_df.columns:
    if col in add_bracket:
        print(col)
        platforms_relations_df[col] = format_cols_for_md(platforms_relations_df[col])
    elif col in replace_bracket:
        platforms_relations_df[col] = replace_chars_add_bracket(platforms_relations_df[col])
    else:
        platforms_relations_df[col] =  replace_chars(platforms_relations_df[col])
platforms_relations_df.columns = platforms_relations_df.columns.str.replace('_', ' ', regex=False)
platforms_relations_df.head(3)

platform


,platform,technique ID,technique
0,[[Azure AD]],[[T1059.009]] [[T1098.001]] [[T1548]] [[T1528]...,"Unsecured Credentials, Conditional Access Poli..."
1,[[Containers]],[[T1068]] [[T1190]] [[T1136.001]] [[T1496]] [[...,"Unsecured Credentials, Use Alternate Authentic..."
2,[[Google Workspace]],[[T1059.009]] [[T1566.004]] [[T1564.008]] [[T1...,"Unsecured Credentials, Use Alternate Authentic..."


In [62]:
for index, row in platforms_relations_df.iterrows():
    # Obtener el platform sin los corchetes
    platform = row['platform'].strip('[]')
    
    # Crear la ruta del archivo
    folder_path = os.path.join(output_folder, 'mitre properties', 'relations_md', mitre_domain, 'platforms_techniques', platform)
    os.makedirs(folder_path, exist_ok=True)
    
    # Crear el nombre del archivo .md
    filename = f'relations_{platform}.md'
    file_path = os.path.join(folder_path, filename)
    
    # Escribir el contenido en el archivo .md
    with open(file_path, 'w') as file:
        # Propiedades
        file.write("---\n")
        file.write("Platform: true\n")
        file.write("Platform relations: true\n")
        file.write("---\n\n")
        
        # Escribir el título con platform
        file.write(f"### {platform}\n\n")
        
        # Escribir los detalles de la táctica
        file.write(f"**platform**\n{row['platform']}\n\n")
        file.write(f"**related techniques**\n{row['technique ID']}\n\n")

        file.close()
    print(f"Archivo '{filename}' creado en la carpeta '{file_path}' con éxito.")

Archivo 'relations_Azure AD.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\platforms_techniques\Azure AD\relations_Azure AD.md' con éxito.
Archivo 'relations_Containers.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\platforms_techniques\Containers\relations_Containers.md' con éxito.
Archivo 'relations_Google Workspace.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\platforms_techniques\Google Workspace\relations_Google Workspace.md' con éxito.
Archivo 'relations_IaaS.md' creado en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\mitre_relationships\outputs\mitre properties\relations_md\enterprise\platforms_techniques\IaaS\relations_IaaS.md' con éxito.
Archivo 'relations_Lin